In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# 1. VERİLERİ OKUMA VE BİRLEŞTİRME
matches_df = pd.read_csv('data/processed/matches.csv')
maps_df = pd.read_csv('data/processed/maps.csv')
player_df = pd.read_csv('data/processed/player_stats.csv')

merged_df = pd.merge(matches_df, maps_df, on='match_id', how='inner')

# --- KRİTİK FİLTRE: unknown verileri temizle ve geçerli listeyi uygula ---
merged_df['map_name'] = merged_df['map_name'].astype(str).str.lower().str.strip()
merged_df = merged_df[merged_df['map_name'] != 'unknown'] # unknown olanları sil
valid_maps = ['ascent', 'bind', 'haven', 'split', 'lotus', 'sunset', 'abyss', 'icebox', 'fracture', 'breeze', 'pearl']
merged_df = merged_df[merged_df['map_name'].isin(valid_maps)]

# Kazananı skorlara göre belirliyoruz
merged_df['team_a_won_map'] = (merged_df['team_a_score'] > merged_df['team_b_score']).astype(int)

# 2. AJANLARI İŞLEME VE DÜZELTME
player_df['agent'] = player_df['agent'].astype(str).str.split(',').str[0].str.strip()

# Ajan One-Hot Encoding
agent_picks = player_df[['map_id', 'team', 'agent']]
agent_dummies = pd.get_dummies(agent_picks, columns=['agent'], prefix='', prefix_sep='')
team_agents = agent_dummies.groupby(['map_id', 'team']).max().reset_index()

# Team A ve Team B ajanlarını ayırma
team_a_agents = team_agents[team_agents['team'] == 'team_a'].drop('team', axis=1)
team_b_agents = team_agents[team_agents['team'] == 'team_b'].drop('team', axis=1)

team_a_agents.columns = [col if col == 'map_id' else f"{col}_t1" for col in team_a_agents.columns]
team_b_agents.columns = [col if col == 'map_id' else f"{col}_t2" for col in team_b_agents.columns]

# Birleştirme (Filtrelenmiş merged_df üzerinden)
merged_df = pd.merge(merged_df, team_a_agents, on='map_id', how='inner')
merged_df = pd.merge(merged_df, team_b_agents, on='map_id', how='inner')

# 3. HARİTALARI İŞLEME (ONE-HOT ENCODING)
map_dummies = pd.get_dummies(merged_df['map_name'], prefix='map')
merged_df = pd.concat([merged_df, map_dummies], axis=1)

# 4. SÜTUNLARI AYIRMA
map_cols = list(map_dummies.columns)
t1_agent_cols = [col for col in merged_df.columns if col.endswith('_t1') and col != 'team_t1']
t2_agent_cols = [col for col in merged_df.columns if col.endswith('_t2') and col != 'team_t2']

# 5. SİMETRİK VERİ ÜRETİMİ
df_normal = merged_df[map_cols + t1_agent_cols + t2_agent_cols].copy()
df_normal['target'] = merged_df['team_a_won_map']

df_swapped = merged_df[map_cols + t2_agent_cols + t1_agent_cols].copy()
df_swapped.columns = map_cols + t1_agent_cols + t2_agent_cols
df_swapped['target'] = 1 - merged_df['team_a_won_map'] 

df_symmetric = pd.concat([df_normal, df_swapped], ignore_index=True)
df_symmetric = df_symmetric.fillna(0)

bool_cols = df_symmetric.select_dtypes(include=['bool']).columns
df_symmetric[bool_cols] = df_symmetric[bool_cols].astype(int)

X = df_symmetric.drop('target', axis=1)
y = df_symmetric['target']

# 6. EĞİTİM VE TEST
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modelin ajanların etkisini daha iyi kavraması için derinliği artırıyoruz
model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=15, 
    min_samples_split=5, 
    random_state=42
)
model.fit(X_train, y_train)

print(f"🚀 Simetrik Model Doğruluk Oranı: %{accuracy_score(y_test, model.predict(X_test)) * 100:.2f}")

joblib.dump(model, 'valorant_rf_model.pkl')
joblib.dump(list(X.columns), 'model_columns.pkl')
print("✅ Model ve Sütunlar Kaydedildi!")

# 7. HARİTAYA ÖZEL AJAN İSTATİSTİKLERİ
map_agent_stats = {}
for m_col in map_cols:
    map_name = m_col.replace('map_', '')
    map_data = merged_df[merged_df[m_col] == 1]
    
    agent_win_rates = {}
    for agent_col in t1_agent_cols:
        agent_name = agent_col.replace('_t1', '')
        agent_played = map_data[map_data[agent_col] == 1]
        
        if len(agent_played) > 3: 
            win_rate = agent_played['team_a_won_map'].mean() * 100
            agent_win_rates[agent_name] = round(win_rate, 1)
            
    sorted_agents = sorted(agent_win_rates.items(), key=lambda item: item[1], reverse=True)[:5]
    map_agent_stats[map_name] = sorted_agents

joblib.dump(map_agent_stats, 'map_agent_stats.pkl')
print("✅ Haritaya Özel Ajan İstatistikleri (map_agent_stats.pkl) Kaydedildi!")

🚀 Simetrik Model Doğruluk Oranı: %82.18
✅ Model ve Sütunlar Kaydedildi!
✅ Haritaya Özel Ajan İstatistikleri (map_agent_stats.pkl) Kaydedildi!
